In [4]:
import kagglehub
path = kagglehub.dataset_download("shivamb/netflix-shows")

Using Colab cache for faster access to the 'netflix-shows' dataset.


In [5]:
import kagglehub

path = kagglehub.dataset_download("shivamb/netflix-shows")
print(f"✅ Downloaded to: {path}")

Using Colab cache for faster access to the 'netflix-shows' dataset.
✅ Downloaded to: /kaggle/input/netflix-shows


In [6]:
import os

for file in os.listdir(path):
    print(file)

netflix_titles.csv


In [7]:
import pandas as pd

df = pd.read_csv(f"{path}/netflix_titles.csv")

print(f"✅ Loaded: {len(df)} titles")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

✅ Loaded: 8807 titles
Columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [8]:
# Keep useful columns for RAG
df = df[['show_id', 'type', 'title', 'director', 'cast', 'country',
         'release_year', 'rating', 'listed_in', 'description']].copy()

# Drop rows with missing description (core of our RAG)
df.dropna(subset=['description'], inplace=True)
df.fillna('Unknown', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"✅ Clean size: {len(df)} titles")
print(f"\nType distribution:\n{df['type'].value_counts()}")

✅ Clean size: 8807 titles

Type distribution:
type
Movie      6131
TV Show    2676
Name: count, dtype: int64


In [9]:
def format_document(row):
    return (
        f"Title: {row['title']}\n"
        f"Type: {row['type']}\n"
        f"Director: {row['director']}\n"
        f"Cast: {row['cast']}\n"
        f"Country: {row['country']}\n"
        f"Release Year: {row['release_year']}\n"
        f"Rating: {row['rating']}\n"
        f"Genre: {row['listed_in']}\n"
        f"Description: {row['description']}"
    )

df['document'] = df.apply(format_document, axis=1)

print("✅ Sample document:\n")
print(df['document'][0])
print("\n" + "="*60)
print(f"\nTotal documents ready: {len(df)}")

✅ Sample document:

Title: Dick Johnson Is Dead
Type: Movie
Director: Kirsten Johnson
Cast: Unknown
Country: United States
Release Year: 2020
Rating: PG-13
Genre: Documentaries
Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.


Total documents ready: 8807


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import json, os

os.makedirs('/content/drive/MyDrive/ecom_rag/data', exist_ok=True)

records = df[['show_id', 'document']].to_dict(orient='records')
save_path = '/content/drive/MyDrive/ecom_rag/data/netflix_cleaned.json'

with open(save_path, 'w') as f:
    json.dump(records, f, indent=2)

print(f"✅ Saved {len(records):,} documents → Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Saved 8,807 documents → Google Drive


In [11]:
import json
from datetime import datetime

def format_mongo_document(row):
    return {
        "_id": row['show_id'],           # MongoDB _id field
        "metadata": {
            "title"       : row['title'],
            "type"        : row['type'],
            "director"    : row['director'],
            "cast"        : row['cast'],
            "country"     : row['country'],
            "release_year": int(row['release_year']) if row['release_year'] != 'Unknown' else None,
            "rating"      : row['rating'],
            "genre"       : row['listed_in'],
        },
        "content": (
            f"Title: {row['title']}\n"
            f"Type: {row['type']}\n"
            f"Director: {row['director']}\n"
            f"Cast: {row['cast']}\n"
            f"Country: {row['country']}\n"
            f"Release Year: {row['release_year']}\n"
            f"Rating: {row['rating']}\n"
            f"Genre: {row['listed_in']}\n"
            f"Description: {row['description']}"
        ),
        "created_at": datetime.utcnow().isoformat()
    }

mongo_docs = [format_mongo_document(row) for _, row in df.iterrows()]

print(f"✅ Total MongoDB documents: {len(mongo_docs)}")
print(f"\n📌 Sample document structure:\n")
print(json.dumps(mongo_docs[0], indent=2))

/tmp/ipykernel_2407/2895216958.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat()


✅ Total MongoDB documents: 8807

📌 Sample document structure:

{
  "_id": "s1",
  "metadata": {
    "title": "Dick Johnson Is Dead",
    "type": "Movie",
    "director": "Kirsten Johnson",
    "cast": "Unknown",
    "country": "United States",
    "release_year": 2020,
    "rating": "PG-13",
    "genre": "Documentaries"
  },
  "content": "Title: Dick Johnson Is Dead\nType: Movie\nDirector: Kirsten Johnson\nCast: Unknown\nCountry: United States\nRelease Year: 2020\nRating: PG-13\nGenre: Documentaries\nDescription: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.",
  "created_at": "2026-04-12T18:49:09.380089"
}


In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/ecom_rag/data', exist_ok=True)

save_path = '/content/drive/MyDrive/ecom_rag/data/netflix_mongo.json'

with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(mongo_docs, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(mongo_docs):,} documents → {save_path}")
print(f"📦 File size: {os.path.getsize(save_path)/1024:.1f} KB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Saved 8,807 documents → /content/drive/MyDrive/ecom_rag/data/netflix_mongo.json
📦 File size: 7908.1 KB


In [13]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 46.3 MB/s eta 0:00:00


In [14]:
import json

with open('/content/drive/MyDrive/ecom_rag/data/netflix_mongo.json', 'r') as f:
    mongo_docs = json.load(f)

print(f"✅ Loaded {len(mongo_docs)} documents")
print(f"\n📌 Sample content preview:")
print(mongo_docs[0]['content'])

✅ Loaded 8807 documents

📌 Sample content preview:
Title: Dick Johnson Is Dead
Type: Movie
Director: Kirsten Johnson
Cast: Unknown
Country: United States
Release Year: 2020
Rating: PG-13
Genre: Documentaries
Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.


In [15]:
# For Netflix dataset each document is already small enough
# No need to split further — one doc = one chunk
chunks = []

for doc in mongo_docs:
    chunks.append({
        "id"      : doc["_id"],        # show_id (e.g. "s1")
        "text"    : doc["content"],    # full text for embedding
        "metadata": doc["metadata"]    # title, genre, etc.
    })

print(f"✅ Total chunks ready: {len(chunks)}")
print(f"\n📌 Sample chunk:")
print(f"  ID    : {chunks[0]['id']}")
print(f"  Text  : {chunks[0]['text'][:150]}...")
print(f"  Title : {chunks[0]['metadata']['title']}")

✅ Total chunks ready: 8807

📌 Sample chunk:
  ID    : s1
  Text  : Title: Dick Johnson Is Dead
Type: Movie
Director: Kirsten Johnson
Cast: Unknown
Country: United States
Release Year: 2020
Rating: PG-13
Genre: Documen...
  Title : Dick Johnson Is Dead


In [16]:
!pip install -q "huggingface_hub>=0.34.0,<1.0"
!pip install -q --upgrade sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.5 MB/s eta 0:00:00


In [17]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

from sentence_transformers import SentenceTransformer
import numpy as np

print("⏳ Loading embedding model...")

# Load with cache only after first download
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    cache_folder='/content/model_cache'
)

texts = [chunk['text'] for chunk in chunks]

print(f"⏳ Generating embeddings for {len(texts)} chunks...")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"✅ Embeddings shape: {embeddings.shape}")

⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Generating embeddings for 8807 chunks...


Batches:   0%|          | 0/138 [00:00<?, ?it/s]

✅ Embeddings shape: (8807, 384)


In [18]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index_with_ids = faiss.IndexIDMap(index)
ids = np.arange(len(embeddings)).astype('int64')
index_with_ids.add_with_ids(embeddings, ids)

print(f"✅ FAISS index built!")
print(f"   Total vectors indexed: {index_with_ids.ntotal}")

✅ FAISS index built!
   Total vectors indexed: 8807


In [19]:
query = "romantic comedy movies"
query_vec = model.encode([query], convert_to_numpy=True)

distances, indices = index_with_ids.search(query_vec, 5)

print(f"🔍 Query: '{query}'\n")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    chunk = chunks[idx]
    print(f"  {rank+1}. {chunk['metadata']['title']}")
    print(f"     Genre : {chunk['metadata']['genre']}")
    print(f"     Score : {dist:.4f}")
    print()

🔍 Query: 'romantic comedy movies'

  1. Valentine's Day
     Genre : Comedies, Romantic Movies
     Score : 0.7966

  2. The New Romantic
     Genre : Comedies, Independent Movies, Romantic Movies
     Score : 0.8109

  3. Girlfriend's Day
     Genre : Comedies, Independent Movies
     Score : 0.8171

  4. Happy Anniversary
     Genre : Comedies, Romantic Movies
     Score : 0.8506

  5. Love, Surreal and Odd
     Genre : Comedies, International Movies, Romantic Movies
     Score : 0.8522



In [20]:
import os

save_dir = '/content/drive/MyDrive/ecom_rag/embeddings'
os.makedirs(save_dir, exist_ok=True)

faiss.write_index(index_with_ids, f'{save_dir}/faiss_index.bin')

with open(f'{save_dir}/chunks_store.json', 'w') as f:
    json.dump(chunks, f, indent=2)

print(f"✅ FAISS index saved  → faiss_index.bin")
print(f"✅ Chunks store saved → chunks_store.json")

✅ FAISS index saved  → faiss_index.bin
✅ Chunks store saved → chunks_store.json
